# Lab 3 - Circuits that decide what to do next

Every circuit so far was a fixed list. Written once, sent off, run start to
finish. The measurement sat at the end because there was nothing left to do with
the answer.

A **dynamic circuit** measures partway through and then uses the bit it got to
choose what happens next, while the other qubits are still alive and coherent.
That one addition is what teleportation, error correction and repeat-until-success
are all built from.

Nothing here is graded and nothing leaves the laptop.

In [ ]:
import math
import warnings

from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

warnings.filterwarnings("ignore")

sim = AerSimulator()
SHOTS = 8192

## Measuring partway through

Measuring collapses. That is not a side effect to work around, it is the tool.

Below, one qubit is put in superposition and measured immediately, and then a
second qubit is flipped to match it using an ordinary `cx`. The two always agree,
because after the measurement qubit 0 has a definite value for the `cx` to copy.

In [ ]:
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.measure(0, 0)  # collapse happens here, not at the end
qc.cx(0, 1)  # and now there is a definite value to copy
qc.measure(1, 1)

print(qc.draw())
counts = sim.run(transpile(qc, sim), shots=SHOTS, seed_simulator=1).result().get_counts()
print("\n", dict(sorted(counts.items())))

Only `00` and `11`. Compare that with exercise 11: the same two outcomes, from a
completely different mechanism. There the qubits were entangled and neither had a
value until the end. Here qubit 0 was collapsed first and its value was copied
classically.

The histograms are identical. **The histogram alone cannot tell you which one you
built**, which is the same lesson exercise 11's README makes about shared coins,
arriving from the other direction.

## Acting on the bit you just read

`cx` copies a qubit onto a qubit. What a dynamic circuit adds is a branch taken on
a **classical** bit, written `with qc.if_test((clbit, value)):`.

Here it is used to build a `reset` by hand: measure, and if the answer was 1, flip
it back. Whatever the qubit was doing, it ends in `|0>`.

In [ ]:
qc = QuantumCircuit(1, 2)
qc.ry(2.0, 0)  # some lopsided superposition, mostly |1>
qc.measure(0, 0)  # what it happened to be
with qc.if_test((qc.clbits[0], 1)):
    qc.x(0)  # only runs on the shots that read 1
qc.measure(0, 1)  # what it is now

counts = sim.run(transpile(qc, sim), shots=SHOTS, seed_simulator=4).result().get_counts()
print(qc.draw())
print("\nkeys read right to left: first measurement, then second")
for key, value in sorted(counts.items()):
    second, first = key[0], key[1]
    print(f"  measured {first} then {second}: {value:>5}")

The second measurement is `0` on every single shot, and the first one is still
split roughly 30/70 the way `ry(2.0)` says it should be.

Note the bit order in the keys. Two classical bits in one register print as one
string, and just like the counts strings in exercise 11 the **rightmost character
is bit 0**, which is the measurement that happened first.

## Why this is not merely convenient

The obvious objection: why not just measure everything at the end and sort it out
in Python?

Because by then the other qubits have collapsed too. A dynamic circuit acts while
the rest of the register is still coherent, and no amount of post-processing can
recover that. Teleportation is the cleanest demonstration.

## Teleportation

Alice has a qubit in some state she does not know. She wants Bob to have that
state. She may send him **two classical bits** and nothing else.

The recipe: Alice and Bob share a Bell pair in advance. Alice interferes her
unknown qubit with her half, measures both, and sends the two bits. Bob applies a
correction chosen by those bits, and ends up holding the state.

In [ ]:
def teleport(theta):
    q = QuantumRegister(3, "q")
    a = ClassicalRegister(1, "a")  # Alice's two bits
    b = ClassicalRegister(1, "b")
    out = ClassicalRegister(1, "out")  # what Bob ends up with
    qc = QuantumCircuit(q, a, b, out)

    qc.ry(theta, 0)  # the unknown state, on Alice's qubit 0
    qc.barrier()
    qc.h(1)  # the shared Bell pair: 1 is Alice's, 2 is Bob's
    qc.cx(1, 2)
    qc.barrier()
    qc.cx(0, 1)  # Alice interferes and measures
    qc.h(0)
    qc.measure(0, a)
    qc.measure(1, b)
    qc.barrier()
    with qc.if_test((a, 1)):  # Bob corrects, using only those two bits
        qc.z(2)
    with qc.if_test((b, 1)):
        qc.x(2)
    qc.measure(2, out)
    return qc


print(teleport(math.pi / 3).draw(fold=90))

In [ ]:
print(f"{'theta':>8}{'Bob measures 1':>17}{'sin^2(theta/2)':>17}")
for theta in (0.0, math.pi / 3, math.pi / 2, 2.0, math.pi):
    counts = (
        sim.run(transpile(teleport(theta), sim), shots=SHOTS, seed_simulator=13)
        .result()
        .get_counts()
    )
    # Three registers print space separated, last declared first: "out b a".
    ones = sum(v for key, v in counts.items() if key.split()[0] == "1")
    print(f"{theta:>8.4f}{ones / SHOTS:>17.4f}{math.sin(theta / 2) ** 2:>17.4f}")

Bob's qubit reproduces the statistics of a state he was never sent, to within a
few thousandths at 8192 shots, for every angle. Raise `SHOTS` and the agreement
tightens, because the only thing separating the two columns is sampling noise.

Two things people get wrong about this, both worth being precise on.

**Nothing travelled faster than light.** Bob's qubit is useless until Alice's two
classical bits arrive, and those go at ordinary speed. Delete the two `if_test`
blocks and rerun: Bob's results become 50/50 regardless of theta, carrying nothing.

**Alice's copy is gone.** Her qubit was measured, and the state is not on her side
any more. That is required, not incidental: two copies would violate no-cloning,
which is the same theorem exercise 11's hints point at when they refuse to call
`cx` a copy.

## The correction that looks unnecessary

Drop the `z` correction and rerun the table above. Every number is still right,
which makes it look like dead code.

It is not. A measurement in the Z basis cannot see what `z` fixes, and exercise 16
is what lets you go and look.

In [ ]:
def teleport_and_measure(theta, corrections, basis):
    q = QuantumRegister(3, "q")
    a, b = ClassicalRegister(1, "a"), ClassicalRegister(1, "b")
    out = ClassicalRegister(1, "out")
    qc = QuantumCircuit(q, a, b, out)
    qc.ry(theta, 0)
    qc.h(1)
    qc.cx(1, 2)
    qc.cx(0, 1)
    qc.h(0)
    qc.measure(0, a)
    qc.measure(1, b)
    if "z" in corrections:
        with qc.if_test((a, 1)):
            qc.z(2)
    if "x" in corrections:
        with qc.if_test((b, 1)):
            qc.x(2)
    if basis == "x":
        qc.h(2)  # exercise 16: rotate, because the machine only measures Z
    qc.measure(2, out)

    counts = sim.run(transpile(qc, sim), shots=SHOTS, seed_simulator=13).result().get_counts()
    ones = sum(v for key, v in counts.items() if key.split()[0] == "1")
    return 1 - 2 * ones / SHOTS


print("ry(theta)|0> has <Z> = cos(theta) and <X> = sin(theta).\n")
header = f"{'theta':>7}{'<Z> both':>10}{'<Z> x only':>12}{'<X> both':>11}{'<X> x only':>12}"
print(header + f"{'want':>8}")
for theta in (math.pi / 3, math.pi / 2, 2.0):
    print(
        f"{theta:>7.4f}"
        f"{teleport_and_measure(theta, 'zx', 'z'):>10.4f}"
        f"{teleport_and_measure(theta, 'x', 'z'):>12.4f}"
        f"{teleport_and_measure(theta, 'zx', 'x'):>11.4f}"
        f"{teleport_and_measure(theta, 'x', 'x'):>12.4f}"
        f"{math.sin(theta):>8.4f}"
    )

Read along a row. Dropping `z` leaves `<Z>` untouched and takes `<X>` to zero.

The state Bob receives without that correction is not the state Alice sent. It
agrees on one axis and has lost the other completely, while a Z-basis histogram,
the only thing the hardware natively produces, reports everything as fine.

This is the strongest argument in the course for exercise 16. An entire axis of a
quantum state can be missing while every number in front of you stays correct.

## What it costs on a real machine

Dynamic circuits are not free. The device has to measure, get the bit back to the
control electronics, decide, and apply a gate, all while the other qubits sit
there decohering. On the hardware from lab 2 the measurement alone eats a fifth of
the shortest-lived qubit's coherence, before the decision and the correction gate
have even happened.

Support is visible in the target, alongside the gates.

In [ ]:
device = FakeManilaV2()
control_flow = sorted(
    name for name in device.target.operation_names if name in {"if_else", "for_loop", "switch_case"}
)
print("control flow this device advertises:", control_flow)

worst_t2 = min(device.qubit_properties(q).t2 for q in range(device.num_qubits))
measure_time = max(device.target["measure"][(q,)].duration for q in range(device.num_qubits))
print(f"\nshortest T2 on the device : {worst_t2 * 1e6:>8.1f} us")
print(f"longest measurement       : {measure_time * 1e6:>8.1f} us")
print(f"one measurement costs      {measure_time / worst_t2:>8.1%} of that qubit's coherence,")
print("and that is before the classical decision and the correction gate.")

That fraction is the reason dynamic circuits are used sparingly and deliberately.
They buy something no static circuit can do, and they are charged for in the
currency the machine has least of.

## Where to go next

- Rerun the teleportation table with `AerSimulator.from_backend(FakeManilaV2())`
  instead of the ideal `sim`, and watch how much of it survives real noise.
- Build repeat-until-success: measure, and if the answer is wrong, undo and try
  again inside a `for_loop`.
- The three-qubit repetition code is the smallest error correction there is, and
  it is a mid-circuit measurement plus two `if_test` blocks away from what you
  already wrote above.